In [ ]:
! pip install pandas
! pip install h5py
! pip install rich
! pip install matplotlib
! pip install ipywidgets
! pip install plotly
! pip install ipympl
! pip install tqdm

In [ ]:
!

In [ ]:
# Read an HDF5 dataset
import numpy as np 
import pandas as pd
import h5py
from rich.console import Console
from rich.tree import Tree
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from tqdm.notebook import tqdm

In [ ]:
# create a console object for rich output
console = Console()

# Function to display the contents of an HDF5 file
display(HTML('''
<style>
.output_scroll {
    display: block;
    max-height: 300px;
    max-width: 100%;
    overflow: auto;
}
</style>
'''))

# visualize file structure

In [ ]:
def build_tree(group, tree, max_groups=3):
    """
    Recursively build a tree structure from the HDF5 group.
    parameters:
    -----------
        group : h5py.Group
            The HDF5 group to build the tree from.
        tree : rich.tree.Tree
            The tree structure to populate.
        max_groups : int, optional
            The maximum number of groups to display.
    """
    group_count = 0
    for key in group.keys():
        branch = tree.add(f"{key}")
        item = group[key]
        if isinstance(item, h5py.Group):
            if group.name == '/':
                group_count += 1
                if group_count > max_groups:
                    tree.add("...")
                    break
            build_tree(item, branch, max_groups)


In [ ]:
# Display the tree structure of the HDF5 file
my_datset = '/data/horse/ws/miha493e-research_project/activediffusion/datasets/qm7x-aqm-gas-noP.hdf5'
dataset_name = 'qm7x-aqm-gas-noP'
with h5py.File(my_datset, 'r') as f:
    tree = Tree(dataset_name)
    build_tree(f, tree, max_groups=3)
    console.print(tree)

In [ ]:
# Display the tree structure of the HDF5 file
my_datset = '/data/horse/ws/miha493e-activediffusion_hanna/datasets/new.hdf5'
dataset_name = 'new'
with h5py.File(my_datset, 'r') as f:
    tree = Tree(dataset_name)
    build_tree(f, tree, max_groups=3)
    console.print(tree)

# visualize properties

## one property distribution

In [35]:
def plot_property_dist(dataset, prop='atNUM'):
    """
    Plot the distribution of a specified property (default is 'atNUM') in the dataset.
    Parameters:
    -----------
        dataset : str
            Path to the HDF5 dataset.
        prop : str, optional
            The property to plot (default is 'atNUM').
    """

    with h5py.File(dataset, 'r') as f:
        property_values = []
        for key in f.keys():
            if isinstance(f[key], h5py.Group):
                for molecule in f[key]:
                    if prop in f[key][molecule]:
                        # Append the number of atoms for each molecule
                        property_values.append(len(f[key][molecule][prop]))
        plt.figure(figsize=(10, 6))
        plt.hist(property_values, bins='fd', edgecolor='black', alpha=0.7)
        plt.xlabel(prop)
        plt.ylabel('Frequency')
        plt.title(f'Distribution of {prop} in Molecules')
        plt.xticks(range(1, max(property_values) + 1))
        plt.xticks(fontsize=5)
        plt.grid(axis='y', alpha=0.75)
        plt.tight_layout()
        plt.show()

In [34]:
file_path = '/data/horse/ws/miha493e-research_project/activediffusion/datasets/qm7x-aqm-gas-noP.hdf5'
# Dropdown for property selection
molecule_prop = [
    'DIP',
    'HLgap',
    'atNUM',
    'atXYZ',
    'eAT',
    'eC',
    'eEE',
    'eH',
    'eKIN',
    'eL',
    'eMBD',
    'eNE',
    'eNN',
    'ePBE0',
    'ePBE0+MBD',
    'eTS',
    'eX',
    'eXC',
    'eXX',
    'mC6',
    'mPOL'
]
prop1_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 1:', style={'description_width': 'initial'})
show_button = widgets.Button(description="Show Plot")
# Output area for displaying data
output1 = widgets.Output()

# Callback function for molecule selection
def on_prop1_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global x_axis
    if change['type'] == 'change' and change['name'] == 'value':
        x_axis = change['new']

   
def on_show_plot(b):
    """
    Callback function to handle the show plot button click.
    """
    global x_axis
    with output1:
        clear_output()
        plot_property_dist(file_path, prop=x_axis)

        

# Connect callbacks
prop1_dropdown.observe(on_prop1_change)
show_button.on_click(on_show_plot)


# Display widgets and output
display(widgets.VBox([widgets.HBox((prop1_dropdown, show_button)), output1]))
x_axis = prop1_dropdown.value

## Two properties distribution

In [36]:
def extract_properties2(file_path, property1, property2):
    """
    Extract arrays for 'HLgap' and 'eMBD' from the specified HDF5 file and dataset.

    Parameters:
    -----------
    file_path : str
        Path to the HDF5 file.
    property1 : str
        The first property to extract (e.g., 'HLgap').
    property2 : str
        The second property to extract (e.g., 'eMBD').

    Returns:
    --------
    tuple
        Two lists containing the extracted values for the specified properties.
    """

    property1_values = []
    property2_values = []

    with h5py.File(file_path, "r") as f:
        for group1_key in f.keys():
            group1 = f[group1_key]
            for geom_key in group1.keys():
                geom_group = group1[geom_key]

                if property1 in geom_group and property2 in geom_group:
                    prop1 = geom_group[property1][()]
                    prop2 = geom_group[property2][()]

                    # Convert to scalar if numpy array with zero dims
                    if hasattr(prop1, 'shape') and prop1.shape == ():
                        prop1 = float(prop1)
                    if hasattr(prop2, 'shape') and prop2.shape == ():
                        prop2 = float(prop2)

                    property1_values.append(prop1)
                    property2_values.append(prop2)

    return property1_values, property2_values

In [37]:
def show_properties2(prop1_name, prop1_values, prop2_name, prop2_values):
    """
    Display the first 10 values of two properties.

    Parameters:
    -----------
    prop1_values : list
        List of values for the first property.
    prop2_values : list
        List of values for the second property.
    """
    plt.figure(figsize=(8,6))
    plt.scatter(prop1_values, prop2_values, s=0.1, alpha=0.1)
    plt.xlabel(prop1_name)
    plt.ylabel(prop2_name)
    plt.title(f"plot of {prop1_name} vs {prop2_name}")
    plt.margins(x=0.2, y=0.2)  # 5% margin on both axes
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"plot_{prop1_name}_vs_{prop2_name}.png", dpi=300)
    plt.show()


In [38]:
file_path = '/data/horse/ws/miha493e-research_project/activediffusion/datasets/qm7x-aqm-gas-noP.hdf5'
# Dropdown for property selection
molecule_prop = [
    'DIP',
    'HLgap',
    'atNUM',
    'atXYZ',
    'eAT',
    'eC',
    'eEE',
    'eH',
    'eKIN',
    'eL',
    'eMBD',
    'eNE',
    'eNN',
    'ePBE0',
    'ePBE0+MBD',
    'eTS',
    'eX',
    'eXC',
    'eXX',
    'mC6',
    'mPOL'
]
prop1_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 1:', style={'description_width': 'initial'})
prop2_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 2:', style={'description_width': 'initial'})
show_button = widgets.Button(description="Show Plot")
# Output area for displaying data
output2 = widgets.Output()

# Callback function for molecule selection
def on_prop1_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global x_axis
    if change['type'] == 'change' and change['name'] == 'value':
        x_axis = change['new']

def on_prop2_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global y_axis
    if change['type'] == 'change' and change['name'] == 'value':
        y_axis = change['new']
        
def on_show_plot(b):
    """
    Callback function to handle the show plot button click.
    """
    global x_axis, y_axis
    with output2:
        clear_output()
        x_values, y_values = extract_properties2(file_path, x_axis, y_axis)
        show_properties2(x_axis, x_values, y_axis, y_values)

        

# Connect callbacks
prop1_dropdown.observe(on_prop1_change)
prop2_dropdown.observe(on_prop2_change)
show_button.on_click(on_show_plot)


# Display widgets and output
display(widgets.VBox([prop1_dropdown, widgets.HBox((prop2_dropdown, show_button)), output2]))
x_axis = prop1_dropdown.value
y_axis = prop2_dropdown.value

## Three properties distribution

In [39]:
def extract_properties3(file_path, property1, property2, property3):
    """
    Extract arrays for 'HLgap' and 'eMBD' from the specified HDF5 file and dataset.

    Parameters:
    -----------
    file_path : str
        Path to the HDF5 file.
    property1 : str
        The first property to extract (e.g., 'HLgap').
    property2 : str
        The second property to extract (e.g., 'eMBD').
    property3 : str
        The third property to extract (e.g., 'atNUM').

    Returns:
    --------
    tuple
        Two lists containing the extracted values for the specified properties.
    """

    property1_values = []
    property2_values = []
    property3_values = []

    with h5py.File(file_path, "r") as f:
        for group1_key in f.keys():
            group1 = f[group1_key]
            for geom_key in group1.keys():
                geom_group = group1[geom_key]

                if property1 in geom_group and property2 in geom_group and property3 in geom_group:
                    prop1 = geom_group[property1][()]
                    prop2 = geom_group[property2][()]
                    prop3 = geom_group[property3][()]

                    # Convert to scalar if numpy array with zero dims
                    if hasattr(prop1, 'shape') and prop1.shape == ():
                        prop1 = float(prop1)
                    if hasattr(prop2, 'shape') and prop2.shape == ():
                        prop2 = float(prop2)
                    if hasattr(prop3, 'shape') and prop3.shape == ():
                        prop3 = float(prop3)

                    property1_values.append(prop1)
                    property2_values.append(prop2)
                    property3_values.append(prop3)

    return property1_values, property2_values, property3_values

In [40]:
def show_properties3(prop1_name, prop1_values, prop2_name, prop2_values, prop3_name, prop3_values):
    """
    Display the first 10 values of two properties.

    Parameters:
    -----------
    prop1_values : list
        List of values for the first property.
    prop2_values : list
        List of values for the second property.
    prop3_values : list
        List of values for the third property.
    -----------
    """
    cleaned_prop3_values = [
        val[0] if isinstance(val, (list, np.ndarray)) else val
        for val in prop3_values
    ]

    prop1_values = np.array(prop1_values).flatten()
    prop2_values = np.array(prop2_values).flatten()

    plt.figure(figsize=(8,6))
    #plt.scatter(prop1_values, prop2_values, c=cleaned_prop3_values, cmap='inferno', s=0.1, alpha=0.1)
    plt.hexbin(
        prop1_values,
        prop2_values,
        C=cleaned_prop3_values,
        gridsize=100,
        alpha=1,
        mincnt=1,
        reduce_C_function=np.max,
        cmap='viridis',
        norm=LogNorm(vmin=1, vmax=100))
    cb = plt.colorbar(label=f'Max {prop3_name} (log scale)') 
    ticks = [16, 20, 40, 70]
    cb.set_ticks(ticks)
    cb.set_ticklabels([str(t) for t in ticks])
    plt.xlabel(prop1_name)
    plt.ylabel(prop2_name)
    plt.title(f"plot of {prop1_name} vs {prop2_name}")
    plt.margins(x=0.2, y=0.2)  # 5% margin on both axes
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.tight_layout()
    plt.savefig(f"plot_{prop1_name}_vs_{prop2_name}.png", dpi=300)
    plt.show()


In [42]:
file_path = '/data/horse/ws/miha493e-research_project/activediffusion/datasets/qm7x-aqm-gas-noP.hdf5'
# Dropdown for property selection
molecule_prop = [
    'DIP',
    'HLgap',
    'atNUM',
    'atXYZ',
    'eAT',
    'eC',
    'eEE',
    'eH',
    'eKIN',
    'eL',
    'eMBD',
    'eNE',
    'eNN',
    'ePBE0',
    'ePBE0+MBD',
    'eTS',
    'eX',
    'eXC',
    'eXX',
    'mC6',
    'mPOL'
]
prop1_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 1:', style={'description_width': 'initial'})
prop2_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 2:', style={'description_width': 'initial'})
prop3_dropdown = widgets.Dropdown(options=molecule_prop, description='Molecule Property 3:', style={'description_width': 'initial'})
show_button = widgets.Button(description="Show Plot")

# Output area for displaying data
output3 = widgets.Output()

# Callback function for molecule selection
def on_prop1_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global x_axis
    if change['type'] == 'change' and change['name'] == 'value':
        x_axis = change['new']

def on_prop2_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global y_axis
    if change['type'] == 'change' and change['name'] == 'value':
        y_axis = change['new']

def on_prop3_change(change):
    """
    Callback function to handle changes in the properties dropdown.
    """
    global z_axis
    if change['type'] == 'change' and change['name'] == 'value':
        z_axis = change['new']

def on_show_plot(b):
    """
    Callback function to handle the show plot button click.
    """
    global x_axis, y_axis, z_axis
    with output3:
        clear_output()
        x_values, y_values, z_values = extract_properties3(file_path, x_axis, y_axis, z_axis)
        show_properties3(x_axis, x_values, y_axis, y_values, z_axis, z_values)

        

# Connect callbacks
prop1_dropdown.observe(on_prop1_change)
prop2_dropdown.observe(on_prop2_change)
prop3_dropdown.observe(on_prop3_change)
show_button.on_click(on_show_plot)


# Display widgets and output
display(widgets.VBox([prop1_dropdown, widgets.HBox((prop2_dropdown, show_button)), prop3_dropdown, output3]))
x_axis = prop1_dropdown.value
y_axis = prop2_dropdown.value
z_axis = prop3_dropdown.value

# Visualize molecules